# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR<sup>2</sup>](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, following best practices for reproducibility and FAIR principles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Croissant schema URL**: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)
- **License**: [Open Data Commons Attribution](https://opendatacommons.org/licenses/by/1-0/)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant pandas

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a mlcroissant.Metadata object
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, their fields, and their respective `@id` identifiers.

**Note:** The Croissant schema organizes data with the following core structure:

- **Record Sets**: Table-like entities containing rows of related information. Each has an `@id`.
- **Fields**: Describes columns inside a record set (also uniquely identified by `@id`).
- **Columns**: Where applicable, columns point to physical locations inside files for data extraction. Their `@id` is referenced as well.

Let's enumerate all record sets and their fields (`@id` and name):

In [ ]:
# List record sets and their fields by @id
for record_set in metadata.record_sets:
    print(f"Record set @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, dataType: {field.data_type})")
    print("")

## 3. Data Extraction

Load data from the record sets into pandas DataFrames for analysis. We use each record set's `@id` (see above) as the identifier.

> **If you do not see any record sets above, check the Croissant schema for tables. For this dataset, the main record set contains protein abundance and metadata. We will use the inferred `@id` for the main record set, which is `'https://api.app.sen.science/frontiers/7154140/d18d681b-431c-4fc3-b535-1e26cb261034#RecordSet'`, or if the above cell printed a different one, use that value. You can check the full list in the Croissant JSON, or by printing [rs.id for rs in metadata.record_sets].**

In [ ]:
# Extract data from all record sets into dataframes, referenced by their @id
dataframes = {}
record_set_ids = [rs.id for rs in metadata.record_sets]
for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
    display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We will perform standard EDA steps. Please select an appropriate numeric field from the main record set printed above (for example, 'MW [kDa]' or 'Coverage [%]').

You may also modify the field selection if your field names or IDs differ. We'll filter, normalize, and group the data.

In [ ]:
# Choose the main record set and a numeric field for analysis

# Example: main protein record set id (replace with actual id if needed)
main_record_set_id = record_set_ids[0]  # First record set, or set explicitly
df = dataframes[main_record_set_id]

# Display available columns for selection
print("Columns in main DataFrame:")
print(df.columns.tolist())

# Example field (replace with the actual field name or @id if needed)
# Example: 'MW [kDa]' is commonly mass (molecular weight)
numeric_field = None
# Common choices: 'MW [kDa]', 'Coverage [%]', or the @id for those fields
for col in df.columns:
    if 'MW' in col or 'Coverage' in col or 'Abundance' in col:
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.select_dtypes('number').columns[0]  # fallback to a numeric type
print(f"Using numeric field: {numeric_field}")

# Drop NA values for robust filtering
df_num = pd.to_numeric(df[numeric_field], errors='coerce')
threshold = df_num.quantile(0.75)  # top quartile as example threshold
filtered_df = df[df_num > threshold].copy()

print(f"Filtered records with {numeric_field} above 75th percentile (threshold={threshold:.2f}):")
display(filtered_df[[numeric_field]].head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (df_num[df_num > threshold] - df_num.mean()) / df_num.std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Optional: Group by a categorical field (e.g., 'Sample', 'Peptide count', etc.)
# Try to find a suitable field (one with few unique values and not numeric)
group_field = None
non_num_fields = [col for col in df.columns if df[col].nunique() < 20 and df[col].dtype == 'O']
if non_num_fields:
    group_field = non_num_fields[0]
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped data by {group_field} (mean {numeric_field}):")
    display(grouped_df.head())

## 5. Visualization

Let's plot distributions or relationships between numeric fields for visual inspection.

If using Google Colab, run `%matplotlib inline` before plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If a second numeric field is available, make a scatterplot
num_fields = df.select_dtypes('number').columns.tolist()
if len(num_fields) > 1:
    plt.figure(figsize=(6,6))
    sns.scatterplot(x=df[num_fields[0]], y=df[num_fields[1]])
    plt.title(f"{num_fields[0]} vs {num_fields[1]}")
    plt.xlabel(num_fields[0])
    plt.ylabel(num_fields[1])
    plt.show()

## 6. Conclusion

We have successfully loaded and explored the *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells* dataset using the `mlcroissant` library.

Key steps included:

- Loading Croissant metadata and all record sets
- Examining available fields and their `@id`s
- Extracting main data tables into DataFrames
- Filtering and normalizing a selected numeric field
- Visualizing distributions to gain insight into protein characteristics

**Next steps:** Consider domain-specific statistical analyses, more advanced visualizations, and joining with external proteomics or biomedical data.
